In [ ]:
### Instalar ollama
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

### Disponer como daemon
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

### Descargar modelo
!ollama pull qwen2.5:14b
!pip install ollama -q

### Inicializar repositorio
!git clone https://github.com/average-peruvian/media-framing.git
!mv media-framing/* .
!rm -rf media-framing
!pip install -e .
!wget https://github.com/average-peruvian/media-framing/releases/download/v0-alpha1/noticias_cuerpo.csv
!wget https://github.com/average-peruvian/media-framing/releases/download/v0-alpha1/noticias_metadata.csv

In [ ]:
!pip install unidecode gensim

from medianalysis.judges import Ollama, RESPONSE_SCHEMA

SYSTEM_PROMPT = """Eres un clasificador experto de noticias en español. Tu tarea es determinar
si un artículo trata algún evento (sea económico, político, social, ambiental, legal o tecnológico)
relacionado al proyecto minero Las Bambas, su empresas encargadas o sus comunidades afectadas.
Descartar las ofertas laborales.

Nombres conocidos de sus empresas gestoras:
- Mineral and Metals Group (MMG)
- Xstrata plc, Glencore Xstrata, Glencore
- Guoxin International Investment Corporation
- CITIC Metal Co.

0 = NO ES UNA NOTICIA DEL PROYECTO MINERO, 1 = SÍ ES UNA NOTICIA DEL PROYECTO MINERO
En empresa_minera escribe ÚNICAMENTE el nombre de la empresa o proyecto minero si se menciona explícitamente, o '' si no.
"""

kwargs = {
    'input_csv': 'noticias_cuerpo.csv',
    'output_csv': f'/content/drive/MyDrive/ant/selection/out/worker.csv',
    'id_col': 'id',
    'total_workers': 6,
    'wid': 0, # IMPORTANTE CAMBIAR
    'system_prompt': SYSTEM_PROMPT,
    'response_schema': RESPONSE_SCHEMA
}

llm = Ollama(**kwargs)
llm.run()

In [ ]:
!git clone https://github.com/average-peruvian/media-framing.git
!mv media-framing/* .
!rm -rf media-framing
!pip install -e .
!pip install unidecode gensim

from medianalysis.distrib import merge_workers

merge_workers(
    f'/content/drive/MyDrive/ant/selection/worker.csv',
    'id',
    '/content/drive/MyDrive/ant/selection/out'
)